# Homework: Documenting Your Code + Testing Your Code

## Problem 1 - Write docstrings

The following functions are missing docstrings. Write Google-style docstrings for each function, including `Args`, `Returns`, and `Raises` sections where appropriate. Make sure to document default values and explain what each parameter means.

In [4]:
import numpy as np

def normalize(data, method="zscore"):
    """Normalize a numeric array using a specified method.

    Args:
        data (array-like): Input numeric data to normalize.
        method (str, optional): Normalization method. Use `"zscore"` for
            standard score normalization or `"minmax"` for min-max scaling.
            Defaults to `"zscore"`.

    Returns:
        numpy.ndarray: Normalized data as a NumPy array.

    Raises:
        ValueError: If `method` is not `"zscore"` or `"minmax"`.
    """
    if method == "zscore":
        return (data - np.mean(data)) / np.std(data)
    elif method == "minmax":
        return (data - np.min(data)) / (np.max(data) - np.min(data))
    else:
        raise ValueError(f"Unknown method: {method}")


def weighted_mean(values, weights=None):
    """Compute the mean of values with optional weights.

    Args:
        values (array-like): Input numeric values.
        weights (array-like, optional): Weights aligned with `values`. If
            `None`, the simple arithmetic mean is returned. Defaults to `None`.

    Returns:
        float: The weighted mean of `values`.

    Raises:
        ValueError: If `weights` is provided and its length does not match
            `values`.
    """
    if weights is None:
        return np.mean(values)
    if len(values) != len(weights):
        raise ValueError("values and weights must have the same length")
    return np.sum(values * weights) / np.sum(weights)


def remove_outliers(data, threshold=3.0):
    """Remove values that deviate from the mean by more than a threshold.

    Args:
        data (array-like): Input numeric data.
        threshold (float, optional): Z-score threshold for keeping values.
            Values with absolute deviation greater than `threshold * std` are
            removed. Defaults to `3.0`.

    Returns:
        numpy.ndarray: Array of values within the threshold.
    """
    mean = np.mean(data)
    std = np.std(data)
    mask = np.abs(data - mean) <= threshold * std
    return data[mask]


## Problem 2 - Add type hints

The following functions have incomplete or missing type hints. Add appropriate type hints for all parameters and return values. Use `|` syntax for union types where a parameter can accept multiple types or return `None`.

In [ ]:
from typing import Sequence, Union, Optional
import numpy as np

def clip_values(
    arr: np.ndarray | Sequence[float],
    lower: float,
    upper: float,
) -> np.ndarray:
    """Clip array values to be within [lower, upper] range."""
    return np.clip(arr, lower, upper)


def find_peaks(
    data: np.ndarray | Sequence[float],
    min_height: float | None = None,
) -> list[int] | None:
    """Find indices where values are local maxima above min_height.

    Returns None if no peaks are found.
    """
    peaks: list[int] = []
    for i in range(1, len(data) - 1):
        if data[i] > data[i - 1] and data[i] > data[i + 1]:
            if min_height is None or data[i] >= min_height:
                peaks.append(i)
    if len(peaks) == 0:
        return None
    return peaks


def summarize(
    data: np.ndarray | Sequence[float],
    stats: Sequence[str],
) -> dict[str, float]:
    """Calculate summary statistics for data.

    Args:
        data: Input array of numeric values.
        stats: List of statistic names to compute.
            Valid options: "mean", "median", "std", "min", "max"

    Returns:
        Dictionary mapping statistic names to computed values.
    """
    result: dict[str, float] = {}
    for stat in stats:
        if stat == "mean":
            result[stat] = float(np.mean(data))
        elif stat == "median":
            result[stat] = float(np.median(data))
        elif stat == "std":
            result[stat] = float(np.std(data))
        elif stat == "min":
            result[stat] = float(np.min(data))
        elif stat == "max":
            result[stat] = float(np.max(data))
    return result


TypeError: unsupported operand type(s) for |: 'type' and '_GenericAlias'

## Problem 3: Identifying Test Types

For each scenario below, identify whether the test being described is a **unit test**, **integration test**, or **regression test**. Briefly explain your reasoning.

**(a)** You write a test that verifies `calculate_variance()` returns 0 for the input `[3.0, 3.0, 3.0]`.

**(b)** After discovering that `fit_model()` crashes when given a dataset with a single row, you fix the bug and add a test with a one-row input.

**(c)** You write a test that loads data from a CSV file, passes it through `clean_data()`, fits a model with `fit_linear_regression()`, and verifies the model's R-squared value is within an expected range.

**(d)** A user reports that `normalize()` returns incorrect values when all input values are negative. After fixing the issue, you add a test with input `[-5.0, -3.0, -1.0]`.

(a) Unit test. It isolates a single function with a simple, deterministic input and output check.

(b) Regression test. It targets a specific bug that was found and fixed to prevent it from reappearing.

(c) Integration test. It exercises multiple components together (file I/O, cleaning, modeling) and checks the combined behavior.

(d) Regression test. It encodes a previously failing case after a bug fix to guard against recurrence.

## Problem 4: Code Review - What's Wrong with These Tests?

Review the following test code and identify at least **four** problems with the test design or implementation. Explain why each is problematic and suggest how to fix it.

In [ ]:
import numpy as np

def test_all_statistics():
    data = [10, 20, 30, 40, 50]

    # Test mean
    assert np.mean(data) == 30

    # Test median
    assert np.median(data) == 30

    # Test standard deviation
    assert np.std(data) > 0

    # Test min and max
    assert np.min(data) == 10
    assert np.max(data) == 50

    # Test sum
    assert np.sum(data) == 150

def verify_variance_positive(arr):
    var = np.var(arr)
    assert var >= 0

def test_correlation():
    x = np.array([1.0, 2.0, 3.0])
    y = np.array([2.0, 4.0, 6.0])
    corr = np.corrcoef(x, y)[0, 1]
    assert corr == 1.0

results = []

def test_append_result():
    global results
    results.append(42)
    assert 42 in results

def test_check_results():
    assert len(results) == 1

1. `results` is mutated in `test_append_result()` and then asserted in `test_check_results()`. Create a local variable in each test.
2. `verify_variance_positive()` won’t run in pytest because it doesn’t start with "test_". Rename it as `test_verify_variance_positive()`.
3. `test_all_statistics()` includes many assertions. A single failure doesn’t localize the issue well and can mask later checks. Split into separate tests (e.g., test_mean, test_median, etc.).
4. `np.std(data) > 0` doesn’t validate the correct value, just that it’s positive. Assert an expected value with `np.isclose()`.

## Problem 5: The Flaky Test

Your colleague wrote the following test for a bootstrap confidence interval function:

In [ ]:
import numpy as np

def bootstrap_ci(data, confidence=0.95, n_bootstrap=1000):
    """Compute bootstrap confidence interval for the mean."""
    means = []
    n = len(data)
    for _ in range(n_bootstrap):
        sample = np.random.choice(data, size=n, replace=True)
        means.append(np.mean(sample))

    alpha = 1 - confidence
    lower = np.percentile(means, 100 * alpha / 2)
    upper = np.percentile(means, 100 * (1 - alpha / 2))
    return lower, upper

def test_bootstrap_ci_contains_true_mean():
    data = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
    true_mean = 5.5
    lower, upper = bootstrap_ci(data)
    assert lower < true_mean < upper

**(a)** The test passes most of the time but occasionally fails. Explain why this test is "flaky" (non-deterministic).

**(b)** Your colleague argues: "The test is correct because a 95% confidence interval should contain the true mean 95% of the time, so occasional failures are expected." Is this a good argument for keeping the test as-is? Why or why not?

**(c)** Rewrite the test to be deterministic and reliable while still meaningfully testing the `bootstrap_ci` function. Your solution should: ensure reproducible results and verify that the confidence interval has reasonable properties.

**(d)** Propose an alternative testing strategy that could verify the 95% coverage property without making the test flaky. You don't need to implement it, but describe the approach.

(a) It’s flaky because `bootstrap_ci()` uses random resampling without a fixed seed. Each run produces a different CI, and with finite samples/bootstraps there’s a non‑zero chance the true mean falls outside the interval.

(b) No. Unit tests should be deterministic and reliable. Statistical properties are fine to validate, but not with a single stochastic assertion.

(c)

In [ ]:
import numpy as np

def test_bootstrap_ci_deterministic():
    data = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

    rng = np.random.default_rng(123)
    lower, upper = bootstrap_ci(data, confidence=0.95, n_bootstrap=2000, rng=rng)

    # Basic properties
    assert lower < upper
    assert lower <= np.mean(data) <= upper

    # Reproducible exact bounds for this seed/config
    assert np.isclose(lower, 3.4, atol=0.2)
    assert np.isclose(upper, 7.6, atol=0.2)


(d) A statistical integration test that runs many independent trials (e.g., 1,000 datasets from a known distribution), computes CIs, and checks the empirical coverage is within a tolerance (e.g., 93–97%). 